In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install pyarrow -q
!cp "/content/drive/MyDrive/NIPS_2026/Notebooks/20_coverage_denominator.py" .

src = open("20_coverage_denominator.py").read()
for marker in ["canon_lab", "merge_asof", "AUDIT — rule drug", "DESIGN DEFECT"]:
    print(("OK   " if marker in src else "STALE"), marker)

In [ ]:
import os
D = "/content/drive/MyDrive/NIPS_2026"
for p in ["Dataset/mimic-iv-hosp/prescriptions.csv.gz",
          "Dataset/mimic-iv-hosp/labevents.csv.gz",
          "Dataset/mimic-iv-hosp/patients.csv.gz",
          "MedConflict-Bench/medconflict_bench_v2.jsonl"]:
    f = f"{D}/{p}"
    print(("OK  " if os.path.exists(f) else "MISSING "), p,
          f"{os.path.getsize(f)/1e9:.2f} GB" if os.path.exists(f) else "")

In [ ]:
!python 20_coverage_denominator.py --stage all --quick \
  --bench-dir "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench" \
  --mimic-dir "/content/drive/MyDrive/NIPS_2026/Dataset/mimic-iv-hosp" \
  --out-dir  "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage"

In [ ]:
!python 20_coverage_denominator.py --stage all \
  --bench-dir "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench" \
  --mimic-dir "/content/drive/MyDrive/NIPS_2026/Dataset/mimic-iv-hosp" \
  --out-dir  "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage"

In [ ]:
!cp "/content/drive/MyDrive/NIPS_2026/Notebooks/22_coverage_diagnostics.py" .
!ls -la 2*.py

In [ ]:
!python 22_coverage_diagnostics.py \
  --mimic-dir "/content/drive/MyDrive/NIPS_2026/Dataset/mimic-iv-hosp" \
  --bench-dir "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench" \
  --cov-dir   "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage/full"

In [ ]:
!python 23_verify_constants.py \
  --mimic-dir "/content/drive/MyDrive/NIPS_2026/Dataset/mimic-iv-hosp" \
  --bench-dir "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench" \
  --cov-dir   "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage/full"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

N = "/content/drive/MyDrive/NIPS_2026/Notebooks"
!cp "{N}/20_coverage_denominator.py" .
!cp "{N}/23_verify_constants.py" .
!ls -la *.py

src = open("20_coverage_denominator.py").read()
for m in ["51301", "NONTHERAPEUTIC", "_itemids", "canon_lab"]:
    print(("OK   " if m in src else "STALE"), m)

In [ ]:
src = open("20_coverage_denominator.py").read()
print("bytes:", len(src))    # expect ~32,100
for m in ["51301", "NONTHERAPEUTIC", "_itemids", "canon_lab",
          "merge_asof", "DESIGN DEFECT"]:
    print(("OK   " if m in src else "STALE"), m)

In [ ]:
!python 23_verify_constants.py \
  --mimic-dir "/content/drive/MyDrive/NIPS_2026/Dataset/mimic-iv-hosp" \
  --bench-dir "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench" \
  --cov-dir   "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage/full" \
  2>&1 | tee "/content/drive/MyDrive/NIPS_2026/Rebuttal_Coverage/verify_constants.txt"

In [ ]:
import json, collections, numpy as np

BENCH = "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/medconflict_bench_v2.jsonl"
B = [json.loads(l) for l in open(BENCH, encoding="utf-8")]

for rid in ["NUM-012", "NUM-015"]:
    c = [e for e in B if e.get("rule_id") == rid]
    print(rid, len(c), "cases")
    if not c:
        continue
    v = [float(e["lab_value"]) for e in c if e.get("lab_value") is not None]
    print("   lab", c[0].get("lab_name"), "op", c[0].get("operator"),
          "thresh", c[0].get("threshold"))
    print("   lab_value  min", min(v), " median", np.median(v), " max", max(v))
    print("   strata", dict(collections.Counter(e["stratum"] for e in c)))
    print("   split", dict(collections.Counter(e.get("split") for e in c)))

s2 = [e for e in B if e.get("stratum") == 2 and e.get("split") == "test"]
bad = [e for e in s2 if e.get("rule_id") in ("NUM-012", "NUM-015")]
print()
print("S2 test cases:", len(s2), " from broken rules:", len(bad),
      f"({len(bad)/max(len(s2),1):.1%})")

In [ ]:
import json, collections

BENCH = "/content/drive/MyDrive/NIPS_2026/MedConflict-Bench/medconflict_bench_v2.jsonl"
B = [json.loads(l) for l in open(BENCH, encoding="utf-8")]

FIX = {"NUM-012": 100.0, "NUM-015": 3.0}   # K/uL
for rid, newt in FIX.items():
    c = [e for e in B if e.get("rule_id") == rid]
    still = [e for e in c if float(e["lab_value"]) < newt]
    flip  = [e for e in c if float(e["lab_value"]) >= newt]
    print(f"{rid}: {len(c)} cases -> {len(still)} remain conflict, "
          f"{len(flip)} MISLABELLED (should be safe)")
    print("   mislabelled values:",
          sorted(round(float(e['lab_value']), 1) for e in flip))
    print("   by split:", dict(collections.Counter(e.get('split') for e in flip)))
    print()

s2t = [e for e in B if e.get("stratum") == 2 and e.get("split") == "test"]
mis = [e for e in s2t if e.get("rule_id") in FIX
       and float(e["lab_value"]) >= FIX[e["rule_id"]]]
print(f"S2 test: {len(s2t)} cases, {len(mis)} mislabelled "
      f"({len(mis)/len(s2t):.1%})")
print(f"corrected S2 denominator: {len(s2t) - len(mis)}")